# Systematic-review screening pipeline

Fill the `include` / `reason` columns of **`25 papers SP.xlsx`** reproducibly, and
**cross-check GPT-5.5 against a local model (qwen3-8b)** to catch the remaining
mistakes.

This notebook is deliberately thin: it *kicks off* the Python scripts in `pipeline/`
and *visualises / verifies* the output. All logic lives in the scripts, so it runs
the same from the terminal, local Jupyter, or Colab.

```
step0_load      xlsx            -> data/step0_papers.jsonl
step1_resolve   DOI             -> canonical URL + title/year checks   (catches wrong links)
step2_fetch     URL             -> raw text (HTML; PDF text-layer; GLM-OCR fallback)
step3_screen    text+metadata   -> include / reason           (run once per backend)
merge_results   all backends    -> 25 papers SP_screened.xlsx  (with disagreement flags)
```

**How the 3 mistakes surface:** Step 1 flags papers whose sheet link is wrong (e.g.
three papers sharing one ScienceDirect URL); the merge flags every paper where the
two models disagree, or where a model disagrees with the current human decision.
Sort the output sheet by `review_priority` and those rows sit at the top.

## Run environment

**Local (this machine):** pick the **`Python (screening)`** kernel (Kernel ▸ Change
kernel) and run top-to-bottom. qwen runs on your remote Ollama box over the mapped
`:11434` port; GPT‑5.5 also runs because your key is in `.env`.

> First run? Create the kernel from the repo's own `.venv` (it's git‑ignored):
> ```
> python3 -m venv .venv
> .venv/bin/pip install -r requirements.txt ipykernel ipywidgets
> .venv/bin/python -m ipykernel install --user --name screening --display-name 'Python (screening)'
> ```
> ⚠️ Keep the checkout OUTSIDE `~/Documents` — iCloud evicts the venv's binaries and
> imports hang. `~/git/…` (where this repo lives) is fine.

**Colab (no edits needed):** open this notebook on a **GPU runtime** and *Run all*. The
setup cell clones the repo, pip‑installs, **installs + starts Ollama on the Colab GPU**,
and downloads the model. Default is **qwen3:8b** (fits a 16 GB T4). GPT‑5.5 is skipped
unless you set `os.environ['OPENAI_API_KEY']='sk-…'` before the backends cell.

Change `QWEN_PRESET` in the setup cell to pick a bigger/newer qwen (each preset shows its
GPU VRAM). Every step runs **in-process**, so each shows a **live progress bar**.

In [ ]:
# --- environment setup: self-contained; identical locally and on Colab (runs on Ollama) ---
import os, sys, subprocess, time, shutil, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# ============================================================================
#  CHOOSE YOUR QWEN MODEL — you're in control; each preset shows the GPU VRAM it
#  needs. It runs on Ollama's OpenAI-compatible API: locally that's your remote
#  24 GB box over the mapped :11434 port; on Colab this cell installs Ollama on the
#  Colab GPU. The default 8b runs anywhere (incl. a 16 GB Colab T4).
#  qwen3.6 is the newest (multimodal, 256K ctx) but only ships at 27b/35b (24/32 GB;
#  it can partly offload on exactly 24 GB). Smaller GPUs use the text-only qwen3.
#      key         Ollama tag       GPU VRAM
QWEN_PRESETS = {
    "8b":      ("qwen3:8b",      6),    # any GPU incl. Colab T4        <-- default
    "14b":     ("qwen3:14b",    11),    # 16 GB
    "27b-3.6": ("qwen3.6:27b",  22),    # 24 GB (may partly offload -> slower)
    "35b-3.6": ("qwen3.6:35b",  28),    # 32 GB
}
QWEN_PRESET = os.environ.get("QWEN_PRESET", "8b")     # <<<<< CHANGE THIS (bigger = better, slower)
_tag, _vram = QWEN_PRESETS[QWEN_PRESET]
# ============================================================================

# One backend everywhere: Ollama's OpenAI-compatible server.
os.environ.setdefault("QWEN_BACKEND", "openai")
os.environ.setdefault("QWEN_BASE_URL", "http://localhost:11434/v1")
os.environ["QWEN_MODEL"] = _tag

def _ollama_up(secs=1, url="http://localhost:11434/api/tags"):
    for _ in range(secs):
        try:
            urllib.request.urlopen(url, timeout=2); return True
        except Exception:
            time.sleep(1)
    return False

def _find_ollama():
    """Locate the ollama binary. `shutil.which` first, then the paths the install
    script uses, then a filesystem search — Colab's Python may not have the install
    dir on its PATH, so a bare `ollama` in Popen raises FileNotFoundError even right
    after a successful install."""
    p = shutil.which("ollama")
    if p:
        return p
    for c in ("/usr/local/bin/ollama", "/usr/bin/ollama", "/bin/ollama",
              "/opt/ollama/bin/ollama", str(Path.home() / ".ollama" / "bin" / "ollama")):
        if Path(c).exists():
            return c
    hit = subprocess.run(
        "find /usr /bin /opt /root -name ollama -type f 2>/dev/null | head -1",
        shell=True, capture_output=True, text=True).stdout.strip()
    return hit or None

# --- self-contained: if the pipeline code isn't here, clone it from GitHub -----------
# So you can drop JUST this notebook anywhere (fresh Colab, a bare folder) and run it.
# The repo also carries the input '25 papers SP.xlsx', so the pipeline can start cold.
REPO = "https://github.com/putssander/systematic-review-pipeline-pharma.git"
REPO_DIR = "systematic-review-pipeline-pharma"
if not Path("pipeline").exists():
    print("pipeline/ not found -> cloning the repo so this notebook is self-contained…")
    if not Path(REPO_DIR).exists():
        subprocess.run(f"git clone -q {REPO}", shell=True)
    if Path(REPO_DIR, "pipeline").exists():
        os.chdir(REPO_DIR)
PROJECT = Path.cwd()

if IN_COLAB:
    # python deps, then install + start Ollama on the Colab GPU (same backend as local)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
    ollama_bin = _find_ollama()
    if ollama_bin is None:
        # The Ollama installer now ships a zstd-compressed tarball; a bare Colab image has
        # no `zstd`, so extraction dies with "This version requires zstd". Install it first.
        if shutil.which("zstd") is None:
            print("installing zstd (needed to unpack Ollama)…")
            subprocess.run("apt-get install -y -q zstd", shell=True, capture_output=True, text=True)
            if shutil.which("zstd") is None:                 # stale apt index -> refresh, retry
                subprocess.run("apt-get update -q && apt-get install -y -q zstd",
                               shell=True, capture_output=True, text=True)
        if shutil.which("zstd") is None:
            raise SystemExit("Could not install 'zstd' (needed to unpack Ollama). Run this in "
                             "a shell cell, then re-run: !apt-get update && apt-get install -y zstd")
        print("installing Ollama… (~30s)")
        # Download the script to a file first, THEN run it. `curl … | sh` hides a failed
        # download: if curl can't reach the CDN, sh runs empty input and exits 0, so the
        # install "succeeds" with no binary. The && makes a fetch failure a real error.
        # capture=True routes output to THIS cell (a bare shell subprocess writes to the
        # kernel log, not here — that's why earlier failures looked silent).
        r = subprocess.run(
            "curl -fsSL https://ollama.com/install.sh -o /tmp/ollama_install.sh "
            "&& sh /tmp/ollama_install.sh",
            shell=True, capture_output=True, text=True)
        if r.stdout.strip():
            print(r.stdout.strip()[-2000:])
        if r.returncode != 0:
            print((r.stderr.strip() or "(no stderr)")[-2000:])
            raise SystemExit(f"Ollama install failed (exit {r.returncode}) — see the "
                             "install script's output above for the reason.")
        ollama_bin = _find_ollama()
    if ollama_bin is None:
        raise SystemExit("Ollama install script ran but the 'ollama' binary wasn't found "
                         "anywhere. Try a fresh runtime, or install it in a shell cell: "
                         "!curl -fsSL https://ollama.com/install.sh | sh")
    print("Ollama binary  :", ollama_bin)
    if not _ollama_up():
        subprocess.Popen([ollama_bin, "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        print("starting Ollama:", "ready" if _ollama_up(90) else "NOT ready — is the runtime a GPU?")

print("Project        :", PROJECT)
print("Colab          :", IN_COLAB)
print("Python (kernel):", sys.executable)
print(f"qwen model     : {_tag}  (preset '{QWEN_PRESET}', needs ~{_vram} GB GPU VRAM)")
print("Ollama         :", "reachable" if _ollama_up() else "NOT reachable at " + os.environ["QWEN_BASE_URL"])

# Guard: a venv inside iCloud Documents gets its packages evicted and hangs on import.
if not IN_COLAB and "/Documents/" in sys.executable:
    print("\n⚠️  This kernel's Python is inside iCloud Documents — packages get evicted and\n"
          "   imports hang. Move the checkout to ~/git and use the 'Python (screening)' kernel.\n")

# Fail fast (instant on the right kernel) if the deps aren't importable.
try:
    import pandas, openai, fitz, bs4, openpyxl, tqdm  # noqa: F401
    print("Deps           : OK (pandas / openai / pymupdf / bs4 / openpyxl / tqdm)")
except Exception as e:
    raise SystemExit(
        f"\nMissing deps: {e}\n"
        "Local: pick the 'Python (screening)' kernel, or create it (its .venv is git-ignored):\n"
        "  python3 -m venv .venv\n"
        "  .venv/bin/pip install -r requirements.txt ipykernel ipywidgets\n"
        "  .venv/bin/python -m ipykernel install --user \\\n"
        "        --name screening --display-name 'Python (screening)'")

In [ ]:
# --- backends (gpt-5.5 optional), model auto-download, in-process step runner ---
import importlib, json, urllib.request
from pipeline import config
importlib.reload(config)            # pick up a QWEN_PRESET/QWEN_MODEL change from the cell above
from pipeline import report

# qwen always runs on Ollama. gpt-5.5 runs ONLY if an OpenAI key is present — from
# .env locally, or set os.environ['OPENAI_API_KEY']='sk-...' before this cell on Colab.
# No prompt, so the notebook never blocks: no key -> qwen-only validation.
BACKENDS = ["gpt-5.5", "qwen"] if os.getenv("OPENAI_API_KEY") else ["qwen"]
print("OpenAI key found -> screening with gpt-5.5 AND qwen, then comparing"
      if "gpt-5.5" in BACKENDS else
      "No OPENAI_API_KEY -> qwen-only run (set a key to also run + compare gpt-5.5)")

def run_step(module, *args):
    """Run a pipeline step *in-process* so its tqdm progress bar streams live here
    (the old `!python -m ...` buffered output and looked frozen)."""
    mod = importlib.import_module(f"pipeline.{module}")
    old_argv = sys.argv
    sys.argv = [module, *[str(a) for a in args]]
    try:
        mod.main()
    except SystemExit as e:
        if e.code not in (0, None):
            print("stopped:", e)
    finally:
        sys.argv = old_argv

def ensure_ollama_model(model):
    """Make sure `model` is on the Ollama server, downloading it once if missing."""
    if config.BACKENDS["qwen"]["kind"] != "openai":
        return
    base = os.environ.get("QWEN_BASE_URL", "http://localhost:11434/v1").rsplit("/v1", 1)[0]
    have = [m["name"] for m in json.load(urllib.request.urlopen(base + "/api/tags", timeout=10))["models"]]
    if model in have or f"{model}:latest" in have:
        print(f"✓ {model} is already on the Ollama server"); return
    print(f"↓ downloading {model} to the Ollama server (one-time)…")
    req = urllib.request.Request(base + "/api/pull",
        data=json.dumps({"name": model}).encode(), headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req) as r:
        for line in r:
            m = json.loads(line)
            if "error" in m:
                raise RuntimeError(m["error"])
            done, tot = m.get("completed"), m.get("total")
            if done and tot:
                print(f"   {m.get('status','')}: {100*done/tot:5.1f}%", end="\r")
    print(f"\n✓ {model} ready")

print("Backends:", BACKENDS, "| qwen ->",
      config.BACKENDS["qwen"]["kind"], config.BACKENDS["qwen"]["model"])

## Step 0 - load the workbook into a clean JSONL

In [ ]:
run_step("step0_load")

In [ ]:
report.papers_df()

## Step 1 - resolve DOIs to the correct URL

Look at the `flags` column. `duplicate_link_shared_with:...` means several papers
carry the *same* link (a data-entry error); `title_mismatch` means the DOI does not
match the title in the sheet. These are the rows whose old decisions are least
trustworthy, because the model may have read the wrong article.

In [ ]:
run_step("step1_resolve")

In [ ]:
# Only the papers with a data-quality problem:
report.flagged_df().style.set_properties(**{'background-color': '#fff3cd'}, subset=['flags'])

## Step 2 - fetch the raw article text

HTML pages are scraped; PDFs use the embedded text layer. OCR runs **only as a last
resort** - a PDF is OCR'd only when it has essentially no text layer (a genuine scan).
OCR is **free and keyless** via local [`glm-ocr` on Ollama](https://ollama.com/library/glm-ocr):

```bash
ollama pull glm-ocr   # one time; then `ollama serve` (usually already running)
```

If Ollama isn't running, Step 2 logs `OCR skipped` and keeps the abstract - it never
stalls. A live progress bar shows ETA; re-run with `--only-missing` to fill just the
gaps, or `--no-ocr` to disable OCR.

**What the `method` / `char_count` columns tell you — and why some rows look empty.**
Paywalled publishers (MDPI, OUP, Wiley, ASCO, IEEE, Elsevier…) block a cloud IP like
Colab's, so expect a chunk of rows to come back as:

| you'll see | meaning |
|---|---|
| `html` / `pdf-text`, thousands of chars | full text scraped ✓ |
| `http_error` | publisher refused the request (403/401) |
| `html` with **0–20 chars** | a 200 response but an empty JS shell (e.g. `linkinghub.elsevier`, IEEE, JMIR) |
| `local-pdf-text` | a PDF **you** supplied was used (see Step 2b) |

**This is not a failure.** Step 3 always feeds the model the **sheet abstract** *plus*
whatever full text we got, so blocked papers are still screened on the abstract — exactly
standard title/abstract screening. Full text is a bonus that helps only the borderline
calls. **Step 2b (next)** is the optional way to upgrade those blocked rows to full text
by downloading their PDFs on a network that has access.

In [ ]:
run_step("step2_fetch_text", "--sleep", 1)

In [ ]:
report.text_df()

## Step 2b (optional) - upgrade the blocked papers to full text

The rows above marked `http_error` or near-empty `html` are paywalled publishers blocking
Colab's IP. They're **already screened on the abstract** — do this only if you want full
text for the borderline calls. It doesn't pull every PDF, just the blocked shortlist, and
the download runs on a network that *has* access:

1. **Run 2b-i** — lists the blocked papers and writes `data/missing_fulltext.csv`
   (auto-downloaded on Colab).
2. On your **university network**, open **[`download_optional_pdfs.ipynb`](download_optional_pdfs.ipynb)**,
   upload that CSV, and run it → it fetches the PDFs and gives you **`manual_pdfs.zip`**.
   (It follows each publisher's `citation_pdf_url`; anything login-walled is listed for a
   quick manual save.)
3. **Run 2b-ii** — upload `manual_pdfs.zip`; Step 2 re-runs and those rows become
   `local-pdf-text`.

Locally (not Colab) it's simpler: drop `<record_id>.pdf` files into `manual_pdfs/` and
re-run 2b-ii.

In [ ]:
# 2b-i — IDENTIFY the blocked/thin papers and export the worklist to fetch elsewhere.
# (They were already screened on the abstract; this just upgrades them to full text.)
from pipeline import report
miss = report.needs_fulltext()
print(f"{len(miss)} paper(s) have blocked/thin full text — the shortlist to fetch via "
      "university access." if len(miss) else "Every paper has usable full text — no Step 2b needed.")
wl = report.write_worklist()
if wl:
    print("worklist ->", wl)
    if IN_COLAB:
        from google.colab import files
        files.download(str(wl))     # hand this CSV to download_optional_pdfs.ipynb
miss

In [ ]:
# 2b-ii — RE-IMPORT: after you built manual_pdfs.zip on the university network, upload it
# here and re-run Step 2. A supplied PDF overrides the blocked page (even under
# --only-missing), so only the papers you provided are re-processed.
import zipfile, shutil
from pipeline import config

if IN_COLAB:
    from google.colab import files
    print("Upload manual_pdfs.zip (or individual <record_id>.pdf files):")
    for name, data in files.upload().items():
        if name.lower().endswith(".zip"):
            open("/tmp/_pdfs.zip", "wb").write(data)
            with zipfile.ZipFile("/tmp/_pdfs.zip") as z:
                for m in z.namelist():
                    if m.lower().endswith(".pdf") and "__MACOSX" not in m:
                        with z.open(m) as s, open(config.PDF_DROP_DIR / Path(m).name, "wb") as o:
                            shutil.copyfileobj(s, o)
        elif name.lower().endswith(".pdf"):
            (config.PDF_DROP_DIR / Path(name).name).write_bytes(data)

staged = sorted(p.name for p in config.PDF_DROP_DIR.iterdir() if p.suffix.lower() == ".pdf")
print(f"{len(staged)} PDF(s) staged in manual_pdfs/: {staged}")
if staged:
    run_step("step2_fetch_text", "--only-missing", "--sleep", 1)
    display(report.text_df())
else:
    print("No PDFs staged. (Locally: just copy <record_id>.pdf files into manual_pdfs/ and re-run.)")

## Step 3a - screen with GPT-5.5 (OpenAI) — optional

Runs only if an OpenAI API key is set (`.env` locally, or `os.environ['OPENAI_API_KEY']`
on Colab). Without a key this step is skipped and you just get the qwen validation.

In [ ]:
if "gpt-5.5" in BACKENDS:
    run_step("step3_screen", "--backend", "gpt-5.5", "--only-missing")
else:
    print("skipped gpt-5.5 (no OPENAI_API_KEY) — running qwen only")

In [ ]:
if "gpt-5.5" in BACKENDS:
    display(report.screen_df('gpt-5.5')[
        ['record_id','decision','include','reason','needs_human_review']])
else:
    print("no gpt-5.5 results (no OPENAI_API_KEY)")

## Step 3b - screen with qwen (local / Colab Ollama)

The model is whatever `QWEN_PRESET` selected in the setup cell (default **qwen3:8b**). It
runs on Ollama — your remote 24 GB box locally, or the Ollama the setup cell installed on
the Colab GPU. The cell below **auto-downloads** the model if it isn't on the server yet,
then screens all 25 papers with a live progress bar (`--only-missing` = resumable).

**Which qwen?** (all Ollama Q4)

| preset | model | GPU VRAM | notes |
|---|---|---|---|
| `8b` | qwen3:8b | ~6 GB | text-only — any GPU / Colab T4 — **default** |
| `14b` | qwen3:14b | ~11 GB | text-only — 16 GB |
| `27b-3.6` | qwen3.6:27b | ~22 GB | newest (multimodal) — 24 GB, may partly offload → slower |
| `35b-3.6` | qwen3.6:35b | ~28 GB | newest — 32 GB |

- **Switched models since your last run?** wipe the results so the column reflects the new
  model: `config.step3_jsonl("qwen").unlink(missing_ok=True)`
- Qwen3's slow `<think>` preamble is disabled (`/no_think`); set `QWEN_NO_THINK=0` + restart
  to re-enable reasoning.

In [ ]:
# Switched models? Uncomment to re-screen all 25 with the new one (else it resumes):
# config.step3_jsonl("qwen").unlink(missing_ok=True)
ensure_ollama_model(os.environ["QWEN_MODEL"])          # download to the server if missing
run_step("step3_screen", "--backend", "qwen", "--only-missing")

In [ ]:
report.screen_df('qwen')[
    ['record_id','model','decision','include','reason','needs_human_review']]

## Step 4 - merge, compare, and verify

`merge_results` writes **`25 papers SP_screened.xlsx`**, sorted so the rows most
likely to contain mistakes are at the top:

| priority | meaning |
|---|---|
| `DISAGREE-MODELS` | GPT-5.5 and qwen3-8b give different include decisions |
| `DISAGREE-HUMAN` | the models agree, but disagree with the current sheet |
| `FLAGGED-STEP1` | wrong/duplicate link or DOI-title mismatch |
| `NEEDS-REVIEW` | a model asked for human review |

In [ ]:
run_step("merge_results", "--backends", *BACKENDS)

In [ ]:
report.summary(BACKENDS)

In [ ]:
report.decision_counts(BACKENDS)

### The shortlist to eyeball

These are your candidate mistakes. Read `*_reason` next to `human_reason` and decide.
Once you agree with a model, copy its `include`/`reason` into the master sheet.

In [ ]:
cmp = report.disagreements(BACKENDS)
cols = ['record_id','short_title','review_priority','human_include'] + \
       [c for c in cmp.columns if c.endswith('_include')] + \
       ['step1_flags']
cmp[cols]

In [ ]:
# Full reasoning for the shortlist (wraps text):
import pandas as pd
pd.set_option('display.max_colwidth', 90)
reason_cols = ['record_id','human_reason'] + [c for c in cmp.columns if c.endswith('_reason')]
cmp[reason_cols]